In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import zipfile
import os
from torch.utils.data import random_split
from torch.utils.data import DataLoader

In [2]:
with zipfile.ZipFile('/content/shapes.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/data/')

In [3]:
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),

])

In [4]:
dataset = datasets.ImageFolder('/content/data', transform=transform)

In [5]:
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

In [6]:
train_dataset, test_dataset = random_split(
    dataset,
    [train_size, test_size]
)

In [7]:
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [8]:
print(len(train_dataset))
print(len(test_dataset))

11976
2994


In [9]:
class ShapeCNN(nn.Module):
    def __init__(self, num_classes):
        super(ShapeCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32 , kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(64, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.classifier = nn.Sequential(
            nn.Linear(64 * 16 * 16, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(),
            nn.Linear(128, num_classes),
        )


    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x


In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [11]:
model = ShapeCNN(num_classes=4).to(device)

In [12]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [13]:
epochs = 5

for epoch in range(epochs):
    model.train()
    running_loss = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Эпоха [{epoch+1}/{epochs}], Потери: {running_loss/len(train_loader):.4f}")

Эпоха [1/5], Потери: 0.1297
Эпоха [2/5], Потери: 0.0139
Эпоха [3/5], Потери: 0.0063
Эпоха [4/5], Потери: 0.0064
Эпоха [5/5], Потери: 0.0025


In [17]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total

print(f"Accuracy: {accuracy:.2f}%")

Accuracy: 98.00%


In [18]:
torch.save(model.state_dict(), 'model.pth')